In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import numpy as np
import statsmodels.api as sm

# Data and setup
df_ss = pd.read_excel(r'D:\Freelancing_Projects\13.Power BI and Pyhton Dashboard\Superstore_Food_Services_Insights.xlsx', sheet_name='Superstore Analysis')
df_ss['Date'] = pd.to_datetime(df_ss['Date'])

columns_conversion = {
    'SalesQuantity': int,
    'StockLevel': int,
    'ReorderLevel': int,
    'CostAmount': float,
    'SalesAmount': float,
    'WasteVolume': float,
    'WasteCost': float,
    'Price': float,
    'DiscountPercentage': float
}

for column, dtype in columns_conversion.items():
    df_ss[column] = df_ss[column].astype(dtype)

monthly_sales = df_ss.groupby(pd.Grouper(key='Date', freq='ME'))['SalesAmount'].sum()
category_sales = df_ss.groupby('Category')['SalesAmount'].sum().sort_values(ascending=False)
dietary_distribution = df_ss['DietaryType'].value_counts()

# Insights Calculations
quarterly_sales = df_ss.groupby(pd.Grouper(key='Date', freq='QE'))['SalesAmount'].sum()
category_sales = df_ss.groupby('Category')['SalesAmount'].sum().sort_values(ascending=False)
dietary_distribution = df_ss['DietaryType'].value_counts()

# Rolling average price
df_ss['Rolling_Avg'] = df_ss.groupby('Date')['Price'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
monthly_rolling_avg = df_ss.groupby(pd.Grouper(key='Date', freq='ME'))['Rolling_Avg'].mean()

# Regression model for Waste Volume vs Sales Amount
X = sm.add_constant(df_ss['SalesAmount']) 
model = sm.OLS(df_ss['WasteVolume'], X).fit()
predictions = model.predict(X)

# Sales after discount calculation
df_ss['SalesAfterDiscount'] = df_ss['SalesAmount'] * (1 - df_ss['DiscountPercentage'] / 100)

# Discount impact
total_sales_without_discount = df_ss['SalesAmount'].sum()
total_sales_after_discount = df_ss['SalesAfterDiscount'].sum()
discount_impact_percentage = (total_sales_after_discount - total_sales_without_discount) / total_sales_without_discount * 100

# Sales Quantity by Dayparts
sales_quantity_dayparts = df_ss.groupby('TimeCategory')['SalesQuantity'].sum().round(1)

# Top 5 Item wise Reorder Level
top_reorder_items = df_ss.groupby('ItemName')['ReorderLevel'].mean().nlargest(5)

# Waste Trend by Dietary Type
waste_trend_dietary = df_ss.groupby([pd.Grouper(key='Date', freq='QE'), 'DietaryType'])['WasteVolume'].sum().reset_index()

# Create a new 'Quarter' column based on the month of the Date
waste_trend_dietary['Quarter'] = waste_trend_dietary['Date'].dt.month

# Map the months to quarters
waste_trend_dietary['Quarter_Label'] = waste_trend_dietary['Quarter'].apply(lambda x: f'Q{(x-1)//3 + 1}')
# -------------------------
# Sales and Profit % Trend over Time
df_ss['Profit'] = df_ss['SalesAmount'] - df_ss['CostAmount']
sales_profit_trend = df_ss.groupby(pd.Grouper(key='Date', freq='QE')).agg({'SalesAmount': 'sum', 'Profit': 'sum'})
sales_profit_trend['ProfitPercentage'] = (sales_profit_trend['Profit'] / sales_profit_trend['SalesAmount']) * 100

# Stock Level and Waste Volume by Item Name as combo chart
stock_waste_by_item = df_ss.groupby('ItemName').agg({'StockLevel': 'sum', 'WasteVolume': 'sum'})

# Categorywise Profit %
category_profit_percentage = df_ss.groupby('Category').agg({'Profit': 'sum', 'SalesAmount': 'sum'})
category_profit_percentage['ProfitPercentage'] = (category_profit_percentage['Profit'] / category_profit_percentage['SalesAmount']) * 100

# Sales by Dietary Type
sales_by_dietary = df_ss.groupby('DietaryType')['SalesAmount'].sum().round(1)

# Sales and Profit % Trend by Item Name
sales_profit_item_trend = df_ss.groupby('ItemName').agg({'SalesAmount': 'sum', 'Profit': 'sum'})
sales_profit_item_trend['ProfitPercentage'] = (sales_profit_item_trend['Profit'] / sales_profit_item_trend['SalesAmount']) * 100

# Quantity and Discount by Item Name
quantity_discount_item = df_ss.groupby('ItemName').agg({'SalesQuantity': 'sum', 'DiscountPercentage': 'mean'}).round(1)


# Dash application
app = Dash(__name__)

# Layout of the dashboard
app.layout = html.Div([
    html.H1('Superstore Food Services Insights Dashboard', style={'text-align': 'center'}),
    
    # 1. Monthly Sales Plot
    html.Div([
        html.H3('Monthly Sales Over Time'),
        dcc.Graph(
            id='monthly-sales-plot',
            figure={
                'data': [
                    go.Scatter(
                        x=monthly_sales.index,
                        y=monthly_sales.values / 1000,
                        mode='lines+markers+text',
                        name='Monthly Sales',
                        text=[f'{y / 1000:.0f}' for y in monthly_sales.values],
                        textposition='top center',
                        marker=dict(color='darkred', size=6),
                    )
                ],
                'layout': go.Layout(
                    title='Monthly Sales Over Time',
                    xaxis={'title': 'Month'},
                    yaxis={'title': 'Sales Amount (K)', 'tickformat': ',.0f'},
                    plot_bgcolor='white',
                    hovermode='closest',
                    showlegend=False
                )
            }
        )
    ], style={'padding': '10px'}),
    
    # 2. Sales by Category Plot
    html.Div([
        html.H3('Sales by Category'),
        dcc.Graph(
            id='category-sales-plot',
            figure={
                'data': [
                    go.Bar(
                        x=category_sales.index,
                        y=category_sales.values / 1000,
                        name='Sales by Category',
                        marker={'color': '#006699'},
                        text=[f'{value / 1000:.0f}' for value in category_sales.values],
                        textposition='outside',
                        textfont=dict(color='#A60227')
                    )
                ],
                'layout': go.Layout(
                    title='Sales by Category',
                    xaxis={'title': 'Category'},
                    yaxis={'title': 'Sales Amount (K)', 'tickformat': ',.0f'},
                    showlegend=False,
                    margin=dict(t=50)
                )
            }
        )
    ], style={'padding': '10px'}),
    
    # 3. Dietary Distribution Pie Chart
    html.Div([
        html.H3('Sales Distribution by Dietary Type'),
        dcc.Graph(
            id='dietary-distribution-pie',
            figure={
                'data': [
                    go.Pie(
                        labels=dietary_distribution.index,
                        values=dietary_distribution.values,
                        hole=0.4
                    )
                ],
                'layout': go.Layout(
                    title='Sales Distribution by Dietary Type'
                )
            }
        )
    ], style={'padding': '10px'}),


     # Sales Quantity by Dayparts Plot
    html.Div([
        html.H3('Sales Quantity by Dayparts'),
        dcc.Graph(
            id='sales-quantity-dayparts',
            figure={
                'data': [
                    go.Bar(
                        x=sales_quantity_dayparts.index,
                        y=sales_quantity_dayparts.values,
                        name='Sales Quantity',
                        marker={'color': '#FFA500'},
                        text=[f'{value}' for value in sales_quantity_dayparts.values],  # Data labels
                        textposition='outside'  # Position of labels
                    )
                ],
                'layout': go.Layout(
                    title='Sales Quantity by Dayparts',
                    xaxis={'title': 'Time Category'},
                    yaxis={'title': 'Sales Quantity'},
                    margin=dict(t=50)
                )
            }
        )
    ], style={'padding': '10px'}),
    
    
    # Top 5 Items by Average Reorder Level
    html.Div([
        html.H3('Top 5 Items by Average Reorder Level'),
        dcc.Graph(
            id='top-reorder-items',
            figure={
                'data': [
                    go.Bar(
                        x=top_reorder_items.index,
                        y=top_reorder_items.values,
                        name='Average Reorder Level',
                        marker={'color': '#33C3F0'},
                        text=[f'{value:.0f}' for value in top_reorder_items.values],
                        textposition='outside',
                        textfont=dict(color='#A60227')
                    )
                ],
                'layout': go.Layout(
                    title='Top 5 Items by Average Reorder Level',
                    xaxis={'title': 'Item Name'},
                    yaxis={'title': 'Average Reorder Level'},
                    margin=dict(t=50),
                    showlegend=False
                )
            }
        )
    ], style={'padding': '10px'}),
    
    # 6. Waste Trend by Dietary Type Quarterly
  html.Div([
    html.H3('Waste Trend by Dietary Type (Quarterly)'),
    dcc.Graph(
        id='waste-trend-dietary',
        figure=px.line(
            waste_trend_dietary,
            x='Quarter_Label',  # Using 'Quarter_Label' instead of 'Date'
            y='WasteVolume',
            color='DietaryType',
            title='Waste Trend by Dietary Type (Quarterly)'
        ).update_layout(
            yaxis_title='Waste Volume',
            xaxis_title='Quarter'
        )
    )
], style={'padding': '10px'}),
    
    # 7. Sales and Profit % Trend over Time Quarterly
    html.Div([
        html.H3('Sales and Profit % Trend over Time'),
        dcc.Graph(
            id='sales-profit-trend',
            figure={
                'data': [
                    go.Bar(
                        x=sales_profit_trend.index,
                        y=sales_profit_trend['SalesAmount'] / 1000,
                        name='Sales Amount',
                        marker=dict(color='#006699'),
                        text=[f'{value / 1000:.0f}' for value in sales_profit_trend['SalesAmount']]  # Data labels
                    ),
                    go.Scatter(
                        x=sales_profit_trend.index,
                        y=sales_profit_trend['ProfitPercentage'],
                        mode='lines+markers+text',
                        name='Profit Percentage',
                        line=dict(color='#E95657'),
                        yaxis='y2',
                        text=[f'{value:.1f}%' for value in sales_profit_trend['ProfitPercentage']],
                        textfont=dict(color='#87DEDE'),
                        textposition='top center'
                    )
                ],
                'layout': go.Layout(
                    title='Sales and Profit % Trend over Time',
                    xaxis={'title': 'Quarter'},
                    yaxis={'title': 'Sales Amount (K)', 'tickformat': ',.0f'},
                    yaxis2=dict(title='Profit Percentage', overlaying='y', side='right'),
                    margin=dict(t=50),
                    showlegend=True
                )
            }
        )
    ], style={'padding': '10px'}),
    
    # 8. Stock Level and Waste Volume by Item Name Combo Chart
    html.Div([
        html.H3('Stock Level and Waste Volume by Item Name'),
        dcc.Graph(
            id='stock-waste-by-item',
            figure={
                'data': [
                    go.Bar(
                        x=stock_waste_by_item.index,
                        y=stock_waste_by_item['StockLevel'],
                        name='Stock Level',
                        marker=dict(color='teal')
                    ),
                    go.Scatter(
                        x=stock_waste_by_item.index,
                        y=stock_waste_by_item['WasteVolume'],
                        mode='lines+markers',
                        name='Waste Volume',
                        line=dict(color='orange'),
                        yaxis='y2'
                    )
                ],
                'layout': go.Layout(
                    title='Stock Level and Waste Volume by Item Name',
                    xaxis={'title': 'Item Name'},
                    yaxis={'title': 'Stock Level'},
                    margin=dict(t=50),
                    yaxis2=dict(title='Waste Volume', overlaying='y', side='right'),
                    showlegend=True
                )
            }
        )
    ], style={'padding': '10px'}),

    
    # 9. Categorywise Profit %
    html.Div([
        html.H3('Categorywise Profit %'),
        dcc.Graph(
            id='category-profit-percentage',
            figure=px.bar(
                category_profit_percentage,
                x=category_profit_percentage.index,
                y='ProfitPercentage',
                title='Categorywise Profit Percentage'
            ).update_layout(
                yaxis_title='Profit Percentage', 
                xaxis_title='Category'
            ).update_traces(text=[f'{value:.1f}%' for value in category_profit_percentage['ProfitPercentage']])  # Data labels
        )
    ], style={'padding': '10px'}),
    
    # 10. Sales by Dietary Type
    html.Div([
        html.H3('Sales by Dietary Type'),
        dcc.Graph(
            id='sales-by-dietary',
            figure={
                'data': [
                    go.Bar(
                        x=sales_by_dietary.index,
                        y=sales_by_dietary.values,
                        name='Sales by Dietary Type',
                        marker={'color': '#FF69B4'},
                        text=[f'{value}' for value in sales_by_dietary.values],  
                        textposition='outside'  
                    )
                ],
                'layout': go.Layout(
                    title='Sales by Dietary Type',
                    xaxis={'title': 'Dietary Type'},
                    yaxis={'title': 'Sales Amount'},
                    margin=dict(t=50)
                )
            }
        )
    ], style={'padding': '10px'}),
    
    
    # 11. Sales and Profit % Trend by Item Name
    html.Div([
        html.H3('Sales and Profit % Trend by Item Name'),
        dcc.Graph(
            id='sales-profit-item-trend',
            figure=px.scatter(
                sales_profit_item_trend,
                x='SalesAmount',
                y='ProfitPercentage',
                size='ProfitPercentage',
                color='SalesAmount',
                hover_name=sales_profit_item_trend.index,
                title='Sales and Profit % Trend by Item Name'
            ).update_layout(yaxis_title='Profit Percentage', xaxis_title='Sales Amount')
        )
    ], style={'padding': '10px'}),
    
    # 12. Quantity and Discount by Item Name
    html.Div([
        html.H3('Quantity and Discount by Item Name'),
        dcc.Graph(
            id='quantity-discount-item',
            figure=px.scatter(
                quantity_discount_item,
                x='SalesQuantity',
                y='DiscountPercentage',
                color='SalesQuantity',
                size='DiscountPercentage',
                hover_name=quantity_discount_item.index,
                title='Quantity and Discount by Item Name'
            ).update_layout(yaxis_title='Discount Percentage', xaxis_title='Sales Quantity')
        )
    ], style={'padding': '10px'})
])

# Run the application
if __name__ == '__main__':
    app.run(debug=True, port=8059)
